# FinSight — Financial Transaction & Risk Analytics
Exploratory analysis of the synthetic FinSight transaction dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../data/transactions.csv', parse_dates=['transaction_date'])
df.head()

## Data quality

In [ ]:
print('Shape:', df.shape)
display(df.isna().sum().sort_values(ascending=False))
display(df.describe(include='all').T)

## Executive KPIs

In [ ]:
total_value=df.transaction_amount_inr.sum()
success_rate=df.transaction_status.eq('Success').mean()*100
suspicious=df.fraud_flag.eq('Yes').sum()
print(f'Total value: ₹{total_value:,.2f}')
print(f'Transactions: {len(df):,}')
print(f'Average transaction: ₹{df.transaction_amount_inr.mean():,.2f}')
print(f'Success rate: {success_rate:.2f}%')
print(f'Suspicious transactions: {suspicious:,} ({suspicious/len(df)*100:.2f}%)')

## Monthly transaction trend

In [ ]:
monthly=(df.set_index('transaction_date').resample('MS').agg(transaction_value=('transaction_amount_inr','sum'),transactions=('transaction_id','count')))
display(monthly)
monthly.transaction_value.plot(figsize=(10,5),marker='o',title='Monthly Transaction Value')
plt.ylabel('INR'); plt.xlabel('Month'); plt.tight_layout(); plt.show()

## Merchant and payment analysis

In [ ]:
category=df.groupby('merchant_category').transaction_amount_inr.agg(['sum','count','mean']).sort_values('sum',ascending=False)
display(category)
payment=df.groupby('payment_method').agg(transaction_count=('transaction_id','count'),transaction_value=('transaction_amount_inr','sum'),failure_rate=('transaction_status',lambda x:x.eq('Failed').mean()*100)).sort_values('transaction_value',ascending=False)
display(payment)

## Suspicious-activity analysis

In [ ]:
risk=df.groupby('fraud_flag').transaction_amount_inr.agg(['count','sum','mean'])
display(risk)
risk_by_category=df.groupby('merchant_category').agg(total=('transaction_id','count'),suspicious=('fraud_flag',lambda x:x.eq('Yes').sum()))
risk_by_category['suspicious_rate_pct']=risk_by_category.suspicious/risk_by_category.total*100
display(risk_by_category.sort_values('suspicious_rate_pct',ascending=False))

### Interpretation
The suspicious-activity field is synthetic and is included for portfolio analysis only. It is not a production fraud or AML decision model.